In [1]:
import pandas as pd
import math
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pickle
from haversine import haversine, Unit
from heapq import heappop, heappush
import geopandas as gpd

# Import customized functions
from merge_bus_subway import merge_subway_bus
from compute_shortest_paths_two_layer import compute_shortest_path


In [2]:
weekday_subway_network = pickle.load(open('subway_network_weekday.pickle', 'rb'))
weekend_subway_network = pickle.load(open('subway_network_weekend.pickle', 'rb'))
weekday_bus_network = pickle.load(open('bus_network_weekday.pickle', 'rb'))
weekend_bus_network = pickle.load(open('bus_network_weekend.pickle', 'rb'))

G_merged_weekday = merge_subway_bus(weekday_subway_network, weekday_bus_network, transfer_distance_threshold=0.3)
G_merged_weekend = merge_subway_bus(weekend_subway_network, weekend_bus_network, transfer_distance_threshold=0.3)

In [3]:
geoloc_nyc = gpd.read_file('geoloc_nyc.shp')
geoloc_nyc['centroid'] = geoloc_nyc.geometry.centroid

C:\Users\baodu\AppData\Local\Temp\ipykernel_31192\3726160264.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  geoloc_nyc['centroid'] = geoloc_nyc.geometry.centroid


In [4]:
# Doctor locations
doctors_pos={}
nyc_doctor = pd.read_csv(r'C:\Users\baodu\Dropbox\Summer_research_2024\nyc_doctor_review.csv', encoding='unicode_escape')
nyc_doctor = nyc_doctor.dropna(subset=['Latitude', 'Longitude','Site_Name'])

for index, row in nyc_doctor.iterrows():
    curr = row['count']
    if isinstance(curr, int) or (isinstance(curr, str) and curr.isnumeric()):
        doctors_pos[row['index']] = (float(row['Latitude']), float(row['Longitude']))
 
 
# Zipcode centroids position   
centroids = geoloc_nyc['centroid'].apply(lambda geom: (geom.y, geom.x)).tolist()
zipcodes = geoloc_nyc['ZCTA5'].tolist()
centroids_pos = {}
for i in range(len(zipcodes)):
    centroids_pos[zipcodes[i]] = centroids[i]
    
    
# print(len(doctors_pos))

C:\Users\baodu\AppData\Local\Temp\ipykernel_31192\1379571985.py:3: DtypeWarning: Columns (0,1,2,5,6,12,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,61,63,65,67,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,134,137,138,139,140,141,142,143,144,145,147,149,154,155,156,157,162,163,164,165,166,168,169,170,171,172,173,174,175,176,177,182,183,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,

In [5]:
# Get the information of the nearest station for each zipcode centroid
def get_nearest_stop_zipcode(centroids_pos, station_info, distance_limit):
    nearby_stop = {}
    for zipcode, centroid_pos in centroids_pos.items():
        curr_best = distance_limit
        for stop_id, stop_pos in station_info.items():
            distance = haversine(centroid_pos, stop_pos, unit=Unit.METERS)
            if distance < curr_best:
                nearby_stop[(zipcode, centroid_pos)] = [(distance, stop_id, stop_pos)]
                curr_best = distance
    return nearby_stop

# Get the information of the nearest station for each doctor
def get_nearest_stop_doctor(doctors_pos, station_info, distance_limit):
    nearby_stop = {}
    for doctor_id, doctor_pos in doctors_pos.items():
        curr_best = distance_limit
        for stop_id, stop_pos in station_info.items():
            distance = haversine(doctor_pos, stop_pos, unit=Unit.METERS)
            if distance < curr_best:
                nearby_stop[(doctor_id, doctor_pos)] = [(distance, stop_id, stop_pos)]
                curr_best = distance
    return nearby_stop


In [6]:
def compute_path_from_zipcode_to_doctor(network, zipcode_info, doctor_info, zipcode_nearest_stations, doctor_nearest_stations, time_limit, walking_speed=1.47):
    curr_best = (None, time_limit)
    for stop_z in zipcode_nearest_stations:
        for stop_d in doctor_nearest_stations:
            path, transit_time = compute_shortest_path(network, stop_z[1], stop_d[1])
            time_to_station =  stop_z[0] / walking_speed /60
            time_to_doctor = stop_d[0] / walking_speed / 60
            total_time = time_to_station + transit_time + time_to_doctor

            if total_time <= curr_best[1]:
                curr_best = (path, total_time)

    if curr_best[0]:
        return curr_best
    else:
        return None
            
            
def access_by_subway_bus(transport_network, accessible_zipcodes, accesible_doctors, time_limit, walking_speed=1.47):
    keys = list(accesible_doctors.keys())
    zipcodes_per_doctor = {key : [] for key in keys}
    
    for doctor_info, nearest_stops_d in accesible_doctors.items():
        for zipcode_info, nearby_stops_z in accessible_zipcodes.items():
            distance = haversine(zipcode_info[1], doctor_info[1], unit=Unit.METERS)
    
            if distance<1000:
                walking_time = distance/walking_speed/60
                zipcodes_per_doctor[doctor_info].append((zipcode_info[0], "walking distance", walking_time))
            else:
                res = compute_path_from_zipcode_to_doctor(transport_network, zipcode_info, doctor_info, nearby_stops_z, nearby_stops_d, time_limit)
                if res:
                    # print(doctor_info[0])
                    zipcodes_per_doctor[doctor_info].append((zipcode_info[0], res[0], res[1]))
        print(f'Finish {doctor_info}')
    return zipcodes_per_doctor

# doctor_mappings = access_by_subway(weekday_subway_network, nearby_stops_zipcodes, nearby_stops_doctors, 40)
# doctor_mappings

In [7]:
station_info = nx.get_node_attributes(G_merged_weekday, 'pos')

# reverse the order of coordinates to lat, lon
for key, val in station_info.items():
    reversed_val = val[::-1]
    station_info[key] = reversed_val


nearby_stations_centroids = get_nearest_stop_zipcode(centroids_pos, station_info, distance_limit=500)
nearby_stations_doctors = get_nearest_stop_doctor(doctors_pos, station_info, 500)
# print(nearby_stations_centroids)
# print(nearby_stations_doctors)


In [8]:
# print(doctor_mappings)
def save_data(doctor_mappings, title):
    extracted_data = []
    for key, values in doctor_mappings.items():
        doctor = key[0]
        for value in values:
            zipcode = value[0]
            travel_type = value[1] if isinstance(value[1], str) else 'multiple'
            travel_time = value[2]
            routes = value[1]
            extracted_data.append([doctor, zipcode, travel_type, travel_time, routes])

    # Creating a DataFrame
    df = pd.DataFrame(extracted_data, columns=['Doctor', 'Zipcode', 'Travel_Type', 'Travel_Time', 'Routes'])

    # Saving the DataFrame to a CSV file
    df.to_csv(title, index=False)


In [20]:
doctor_zipcode_travel = access_by_subway_bus(G_merged_weekday, nearby_stations_centroids, nearby_stations_doctors, np.inf)
save_data(doctor_zipcode_travel, 'accessibility_all.csv')

Finish ('44', (40.764381, -73.986807))
Finish ('1420', (40.764285, -73.976729))
Finish ('1439', (40.764432, -73.979927))
Finish ('18167', (40.76017, -73.98371))
Finish ('19727', (40.767973, -73.986206))
Finish ('20031', (40.764381, -73.986807))
Finish ('26089', (40.767001, -73.981597))
Finish ('31933', (40.767818, -73.98372))
Finish ('37486', (40.767818, -73.98372))
Finish ('212', (40.759816, -73.986432))
Finish ('4546', (40.754395, -73.982674))
Finish ('32229', (40.755297, -73.981575))
Finish ('35176', (40.757701, -73.991285))
Finish ('284', (40.622997, -74.16569))
Finish ('311', (40.824703, -73.892146))
Finish ('8386', (40.823104, -73.895278))
Finish ('10137', (40.824703, -73.892146))
Finish ('351', (40.70176, -73.76569))
Finish ('487', (40.845605, -73.937858))
Finish ('2869', (40.851071, -73.935081))
Finish ('10275', (40.847724, -73.938179))
Finish ('18910', (40.847724, -73.938179))
Finish ('19878', (40.845024, -73.938247))
Finish ('23390', (40.848342, -73.939125))
Finish ('36183', 